# 05 — Optional benchmarks: LSTM and TimeGPT

The core of this project deliberately runs on classical and tree-based models:
they are cheap, robust and sufficient for the demand structure shown in
notebooks 02-04. This notebook documents the two optional extensions, why they
are *optional*, and what would justify enabling them.

| Candidate | What it could add | What it costs |
|---|---|---|
| LSTM (PyTorch) | Sequence memory that might capture episode persistence | Heavy dependency, tuning effort, seed sensitivity, harder MLOps |
| TimeGPT (Nixtla API) | Zero-shot forecasts from a foundation model; quick external benchmark | External API dependency, per-call cost, data leaves the environment |

Neither is installed in the base environment, and the project must remain
fully functional without them — the cells below degrade gracefully.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (11, 4)

In [2]:
from clinic_forecast.data import generate_network_data

data_path = PROJECT_ROOT / "data" / "processed" / "clinic_daily_usage.csv"
if data_path.exists():
    usage = pd.read_csv(data_path, parse_dates=["date"])
else:
    usage = generate_network_data().usage
    usage["date"] = pd.to_datetime(usage["date"])

## When would these earn their complexity?

- **LSTM:** if error analysis showed long-memory effects that lag/rolling
  features miss (e.g. episode dynamics with consistent build-up shapes). The
  error anatomy in notebook 04 does not currently show this — most residual
  error is onset surprise, which sequence models cannot see either.
- **TimeGPT:** as an external sanity check — if a zero-shot foundation model
  matched the tuned global model, that would question the feature pipeline;
  if it lagged far behind, that supports the value of domain features. For a
  healthcare network, sending demand data to an external API also requires a
  governance conversation that a PoC should flag, not gloss over.

In [3]:
try:
    import torch

    print(f"PyTorch {torch.__version__} is installed; an LSTM baseline can be added here.")
except ImportError:
    print("PyTorch is not installed (optional group). The LSTM benchmark is skipped.")

PyTorch is not installed (optional group). The LSTM benchmark is skipped.


In [4]:
import os

try:
    from clinic_forecast.models.optional_timegpt import timegpt_forecast  # noqa: F401

    if os.getenv("NIXTLA_API_KEY"):
        print("Nixtla SDK and API key found - TimeGPT benchmarking is available.")
    else:
        print("Nixtla SDK importable but NIXTLA_API_KEY is not set; see .env.example.")
except ImportError:
    print("Nixtla SDK is not installed (optional group). TimeGPT benchmark is skipped.")

Nixtla SDK importable but NIXTLA_API_KEY is not set; see .env.example.


## Status

Both integrations have wrappers in `clinic_forecast.models` with guarded
imports and clear error messages. The roadmap schedules them as benchmarks
once the core pipeline (intervals, staffing optimisation, batch inference) is
complete — accuracy work should follow decision-layer work, not precede it,
because staffing quality is currently limited by uncertainty handling rather
than point-forecast accuracy.